In [ ]:
from dataclasses import dataclass
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import numpy as np 
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
from tqdm.auto import tqdm
from torchfuzzy import FuzzyLayer, DefuzzyLinearLayer, FuzzyBellLayer, DefuzzyMaxLayer
import piqa
import sklearn.metrics as metrics
from sklearn.manifold import TSNE
from torchvision.transforms import v2
from torchinfo import summary
from matplotlib.colors import ListedColormap


In [ ]:
batch_size = 256
learning_rate_ae = 1e-4
num_epochs_ae = 200
latent_dim = 32
mnist_class_anomaly = -1
kernels = 8
fuzzy_rules_count = 10

prefix = f"fuzzy_unsup_clustering"
writer = SummaryWriter(f'runs/mnist/{prefix}_{datetime.now().strftime("%Y%m%d-%H%M%S")}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

binary_cmap = ListedColormap(['yellow', 'red'], N=2)
ssim = piqa.SSIM(window_size = 11, n_channels=1, reduction='none').to(device)
device

## Датасет

1. Исключаем класс аномалии `mnist_class_anomaly` из общей выборк
2. Убираем метки с остальных классов
   

In [ ]:
def norm_and_transform(x):
    nimg = x.view(-1, 28, 28)
    nimg = torch.clamp(nimg, 0, 1)
    return nimg

def clamp(x):
    #nimg = 2.0*(x.view(-1, 28, 28) - 0.5)
    nimg = torch.clamp(x, 0, 1)
    return nimg

transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Lambda(norm_and_transform)
])

augmentation = transforms.Compose([
    transforms.RandomRotation(15, fill=0), 
    transforms.RandomAffine(degrees=5, translate=(0.1, 0.1), fill=0), 
    #transforms.RandomCrop(size=26),
    #transforms.Resize(size=(28, 28)),
    transforms.Lambda(clamp)
])

In [ ]:
class MNISTDatasetWithIdx(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self._dataset = dataset

    def __len__(self):
        return len(self._dataset)

    def __getitem__(self, idx):
        return (*self._dataset[idx], idx)

In [ ]:
def get_target_and_mask(target_label):
    t = target_label
    return t 

train_data = MNISTDatasetWithIdx(datasets.MNIST(
    '~/.pytorch/MNIST_data/', 
    download=True, 
    train=True, 
    transform = transform,
    target_transform = transforms.Lambda(lambda x: get_target_and_mask(x))
))

idx = (train_data._dataset.targets != mnist_class_anomaly)
train_data._dataset.targets = train_data._dataset.targets[idx]
train_data._dataset.data = train_data._dataset.data[idx]
len(train_data)

загружаем тестовую выборку

In [ ]:
test_data = datasets.MNIST(
    '~/.pytorch/MNIST_data/', 
    download=True, 
    train=False, 
    transform=transform, 
    target_transform = transforms.Lambda(lambda x: get_target_and_mask(x))
)
len(test_data)

Создаем итераторы датасетов

In [ ]:

train_loader = torch.utils.data.DataLoader(
    train_data, 
    batch_size=batch_size, 
    shuffle=True,
    
)
test_loader = torch.utils.data.DataLoader(
    test_data, 
    batch_size=batch_size, 
    shuffle=False,
)

In [ ]:
for data,_,_ in iter(train_loader):
    R, C = 1, 2
    plt.subplot(R, C, 1)
    plt.imshow(data[0].squeeze())
    plt.subplot(R, C, 2)
    plt.imshow(augmentation(data)[0].squeeze())
    
    
    break

## Модель

In [ ]:
class Encoder(nn.Module):
    """
    Компонент энкодера для VAE
    
    Args:
        latent_dim (int): Размер латентного вектора.
    """
    
    def __init__(self, latent_dim, fuzzy_rules_count, kernels):
        super(Encoder, self).__init__()
                
        self.encoder = nn.Sequential(
            nn.Conv2d(1, kernels, kernel_size = 5), 
            nn.Conv2d(kernels, kernels, kernel_size = 5), 
            nn.BatchNorm2d(kernels), 
            nn.SiLU(),

            nn.Conv2d(kernels, 2*kernels, kernel_size = 5), 
            nn.Conv2d(2*kernels, 2*kernels, kernel_size = 5), 
            nn.BatchNorm2d(2*kernels), 
            nn.SiLU(),

            nn.Conv2d(2*kernels, 4*kernels, kernel_size = 5), 
            nn.Conv2d(4*kernels, 4*kernels, kernel_size = 5), 
            nn.BatchNorm2d(4*kernels), 
            nn.SiLU(),

            nn.Conv2d(4*kernels, 8*kernels, kernel_size = 4), 
            nn.BatchNorm2d(8*kernels), 
            nn.SiLU(),            
            nn.Flatten(),
            nn.Linear(8*kernels, latent_dim)
        )

        initial_centroids = 1e-10 * np.random.random((fuzzy_rules_count, latent_dim))
        initial_scales = 1 * np.ones((fuzzy_rules_count, latent_dim))
        self.fuzzy = FuzzyLayer.from_centers_and_scales(initial_centroids, initial_scales, trainable=True)
        

    def forward(self, x):
        """
        Выход энкодера для чистого VAE.
        
        Args:
            x (torch.Tensor): Входной вектор.
            eps (float): Небольшая поправка к скейлу для лучшей сходимости и устойчивости.
        
        Returns:
            mu, logvar, z, dist
        """
        mu = self.encoder(x)
        fz = self.fuzzy(mu)
        
        return mu, fz

inp = torch.rand(10, 1, 28, 28)
m = Encoder(latent_dim, fuzzy_rules_count, 16)
#mu, fz, c  = m.forward(inp)

summary(m, input_size=(batch_size, 1, 28, 28))


In [ ]:
class Decoder(nn.Module):
    """
    Компонент декодера для VAE
    
    Args:
        latent_dim (int): Размер латентного вектора.
    """
    
    def __init__(self, latent_dim, kernels):
        super(Decoder, self).__init__()

        self.decode = nn.Sequential(
            nn.Linear(latent_dim, 8*kernels),
            nn.BatchNorm1d(8*kernels),
            nn.Unflatten(1, (8*kernels, 1, 1)),
            
            nn.ConvTranspose2d(8*kernels, 4*kernels, 4),
            nn.BatchNorm2d(4*kernels),
            nn.SiLU(),

            nn.ConvTranspose2d(4*kernels, 2*kernels, 5),
            nn.ConvTranspose2d(2*kernels, 2*kernels, 5),
            nn.BatchNorm2d(2*kernels),
            nn.SiLU(),

            nn.ConvTranspose2d(2*kernels, kernels, 5),
            nn.ConvTranspose2d(kernels, kernels, 5),
            nn.BatchNorm2d(kernels),
            nn.SiLU(),

            nn.ConvTranspose2d(kernels, kernels, 5),
            nn.ConvTranspose2d(kernels, 1, 5),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        
         
    def forward(self, mu):
        """
        Декодирует латентный вектор в исходное представление
        
        Args:
            z (torch.Tensor): Латентный вектор.
        
        Returns:
            x
        """
        out = self.decode(mu)
        return out
    
    
fz = torch.rand(batch_size, latent_dim)
#torch.stack((z,fz), dim=1).unsqueeze(1).shape
m = Decoder(latent_dim, 16)
mu = m.forward(fz)
mu.shape

#summary(m, input_size=(batch_size, latent_dim))

In [ ]:
class VAEFuzzyCFR(nn.Module):
    """
    Args:
        latent_dim (int): Размер латентного вектора.
    """
    def __init__(self, latent_dim, fuzzy_rules_count, kernels):
        super(VAEFuzzyCFR, self).__init__()

        self.encoder = Encoder(latent_dim, fuzzy_rules_count, kernels)        
        self.decoder = Decoder(latent_dim, kernels)
        
    def forward(self, x):
        """
        
        """
        mu, fz = self.encoder(x)
        x_recon = self.decoder(mu)
        
        return mu, fz, x_recon
    
    def half_pass(self, x):
        mu, fz = self.encoder(x)
        return mu, fz
    
    def decoder_pass(self, mu):
        r = self.decoder(mu)
        return r

In [ ]:
fvae = VAEFuzzyCFR(latent_dim=latent_dim, fuzzy_rules_count=fuzzy_rules_count, kernels=kernels).to(device)

num_params = sum(p.numel() for p in fvae.parameters() if p.requires_grad)
print(f'Number of parameters: {num_params:,}')

summary(fvae, input_size=(batch_size, 1, 28, 28))

## Losses

In [ ]:
def compute_ae_loss(x, recon_x):
    
    diff = ssim(x, recon_x)
    loss_recon = (1 - diff).square().mean() 
    return loss_recon

In [ ]:
def keep_eigenvals_positive_loss(layer, eps = 1e-10):
    ev = layer.get_transformation_matrix_eigenvals().real.min()
    ev = torch.clamp(ev, max=eps)
    return -ev

In [ ]:
def fuzzy_clustering_loss(fz):
    winners = torch.argmax(fz, -1)
    winners_mask = F.one_hot(winners, fuzzy_rules_count)

    return (fz - winners_mask).square().mean()

#fz = torch.rand((5, 10))
#fuzzy_clustering_loss(fz)

In [ ]:
def get_arate(model, imgs):
    mu, fz = model.half_pass(imgs)  
    # fz_max = fz.max(-1).indices
    # c = model.fuzzy.get_centroids()
    # c = c[fz_max]
    # x_rec = model.decoder_pass(c)
    
    return fz.sum(-1).detach().cpu().numpy() #torch.cdist(mu, fzc).min(-1).values.detach().cpu().numpy() #compute_sample_grads(model, imgs)#fz[:, 1].cpu().numpy() ##(1 - ssim(x_recon.clamp(0, 1), inp)).cpu().numpy()#xent_continuous_ber((x_recon + 1)/2, (inp + 1)/2).cpu().numpy()# #ssim((inp + 1)/2, (recon_x+1)/2).cpu().numpy() #fz.sum(-1).cpu().numpy()#


## Train AE

In [ ]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']
    
def train(model, dataloader, optimizer, sched, prev_updates, epoch, writer=None):
    model.train()  
    
    for batch_idx, (data, _, item_idx) in enumerate(tqdm(dataloader, disable=True)):
        data = data.to(device)
        adata = augmentation(data)
        
        optimizer.zero_grad()  
        
        mu, fz = model.half_pass(adata)  
        recon_x = model.decoder_pass(mu)
        
        loss = compute_ae_loss(data, recon_x) + fuzzy_clustering_loss(fz)
        #loss_fz = (1 - fz.sum(-1)).square().sum()
        
        ev_loss = keep_eigenvals_positive_loss(model.encoder.fuzzy)
        if ev_loss.item() > 0:
            print("ev singularity")
            ev_loss.backward(retain_graph=True)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1e-2)
        optimizer.step()  
        
        if sched is not None:
            sched.step()
        
    if writer is not None:
        writer.add_scalar('FCL/LR', get_lr(optimizer), global_step=epoch)
        
    return prev_updates + len(dataloader)

In [ ]:
#fixed_random_z = torch.randn(16, fuzzy_rules_count).to(device)

def test(model, dataloader, cur_step, epoch, writer=None):
    model.eval() 

    test_recon_loss = 0
    #test_fz_vol = 0
    test_fz_sum = 0
    #test_fz_cfr_loss = 0
    
    lab_true = []
    lab_pred = []

    #with torch.no_grad():
    for data, lab in tqdm(test_loader, desc='Test MNIST', disable=True):
        data = data.to(device)
        rates = get_arate(model, data)
        
        for f, l in  zip(rates, lab):
            lab_pred.append(f)        
            if l == mnist_class_anomaly:
                lab_true.append(1)
            else:
                lab_true.append(0)
                    
    fpr, tpr, _ = metrics.roc_curve(lab_true, lab_pred)
    roc_auc = metrics.auc(fpr, tpr)
    
    embedings = []
    labels_expected = []
    
    
    with torch.no_grad():
        for data, target in tqdm(dataloader, desc='Testing', disable=True):
            data = data.to(device)

            mu, fz = model.half_pass(data)  
            
            recon_x = model.decoder_pass(mu)
            
            embedings.append(mu.cpu().numpy())
            labels_expected.append((target == mnist_class_anomaly).cpu().numpy())

            loss_recon = compute_ae_loss(data, recon_x)
            
            #centroids = model.fuzzy[0].get_centroids().detach()
            #centroids_imgs = model.decoder_pass(centroids)
            loss_cfr = torch.zeros(1) # compute_cfr_loss(data, recon_x, fz)
            #fz_vol = fuzzy_term_volume_loss(model.encoder.fuzzy)
            fz_sum = fz.sum(-1).mean()
            
            test_recon_loss += loss_recon.item()
            #test_fz_vol += fz_vol.item()
            test_fz_sum += fz_sum.item()
            #test_fz_cfr_loss += loss_cfr.item()
            

    embedings = np.concatenate(embedings, axis=0)
    labels_expected = np.concatenate(labels_expected, axis=0)
   
    test_recon_loss /= len(dataloader)
    #test_fz_vol /= len(dataloader)
    test_fz_sum /= len(dataloader)
    #test_fz_cfr_loss /= len(dataloader)
    
    print(f'[{cur_step}] Reconstruction loss: {test_recon_loss:.4f}, SUM: {test_fz_sum:.2f} AUC: {roc_auc:.4f}')
    
    if writer is not None:
        writer.add_scalar('FCL/AUC', roc_auc, global_step=cur_step)
        writer.add_scalar('FCL/Reconstruction', test_recon_loss, global_step=cur_step)
        #writer.add_scalar('FCL/Volume', test_fz_vol, global_step=cur_step)
        writer.add_scalar('FCL/Sum', test_fz_sum, global_step=cur_step)
        #writer.add_scalar('FCL/CFR', test_fz_cfr_loss, global_step=cur_step)
        
        fig, ax = plt.subplots(1, 2, figsize=(9, 4))
        centroids_c = model.encoder.fuzzy.get_centroids().detach().cpu().numpy()
        
        ax[0].scatter(embedings[:, 0],      embedings[:,  1], c=labels_expected, cmap=binary_cmap, s=2)
        ax[0].scatter(centroids_c[:, 0],      centroids_c[:, 1], marker='1', c='green', s= 50)
        
        #ax[0].set_xlim(-1, 1)
        #ax[0].set_ylim(-1, 1)
        
        ax[1].scatter(embedings[:, 0],      embedings[:,  2], c=labels_expected, cmap=binary_cmap, s=2)
        ax[1].scatter(centroids_c[:, 0],      centroids_c[:, 2], marker='1', c='green', s= 50)
        
        #ax[1].set_xlim(-1, 1)
        #ax[1].set_ylim(-1, 1)
        
        #act_fz = torch.diag(torch.ones(fuzzy_rules_count)).to(device)
        samples = model.decoder_pass(model.encoder.fuzzy.get_centroids().detach())
        img_idx = 0
        fign, axn = plt.subplots(8, 1 + fuzzy_rules_count//8, figsize=(1 + fuzzy_rules_count//8, 8), squeeze=False)
        for i in range(8):
            if img_idx >= fuzzy_rules_count:
                continue
            for j in range(fuzzy_rules_count//8):
                axn[i, j].imshow(samples[img_idx].view(28, 28).cpu().detach().numpy(), cmap='gray')
                axn[i, j].axis('off')
                img_idx += 1

        writer.add_figure('FCL/Emedding', fig, global_step=cur_step)
        writer.add_figure('FCL/Samples', fign, global_step=cur_step)

In [ ]:
prev_updates = 0
optimizer_ae = torch.optim.Adam(fvae.parameters(), lr=learning_rate_ae)
#sched = torch.optim.lr_scheduler.OneCycleLR(optimizer_ae, learning_rate_ae, epochs=num_epochs_ae, steps_per_epoch=len(train_loader))
sched = torch.optim.lr_scheduler.ConstantLR(optimizer_ae, learning_rate_ae)

In [ ]:
for epoch in range(num_epochs_ae):        
    prev_updates = train(fvae, train_loader, optimizer_ae, sched, prev_updates, epoch, writer=writer)
    test(fvae, test_loader, prev_updates, epoch, writer=writer)
    #scheduler.step()

## Визуализируем результаты

In [ ]:
fvae.eval()

In [ ]:
def get_arate_val(inp):
    mu, fz, c = fvae.half_pass(inp)  
    #fz = fvae.fuzzy_pass(mu)
    #recon_x = model.decoder_pass(z)
    return fz.sum(-1).cpu().numpy() ##(1 - ssim(x_recon.clamp(0, 1), inp)).cpu().numpy()#xent_continuous_ber((x_recon + 1)/2, (inp + 1)/2).cpu().numpy()# #ssim((inp + 1)/2, (recon_x+1)/2).cpu().numpy() #fz.sum(-1).cpu().numpy()#

In [ ]:
firings_mnist = {}
firings_mnist['MNIST'] = []
firings_mnist['DISSIDENT'] = []

with torch.no_grad():
    for data, target in tqdm(test_loader, desc='MNIST HIST'):
        data = data.view((-1,1,28,28)).to(device)
        rates = get_arate_val(data)
        for f, l in  zip(rates, target):
            if l != mnist_class_anomaly:
                firings_mnist['MNIST'].append(f)
            else:
                firings_mnist['DISSIDENT'].append(f)
        

labels, data = firings_mnist.keys(), firings_mnist.values()

fig = plt.figure(figsize =(12, 2))
plt.boxplot(data, notch=True, showfliers=False)
plt.xticks(range(1, len(labels) + 1), labels)
plt.show()

writer.add_figure('Anomaly Detection', fig)

In [ ]:
with torch.no_grad():
    firing_levels = []
    lab_true = []
    lab_pred = []

    for data, lab in tqdm(test_loader, desc='Test MNIST', disable=True):
        data = data.view((-1,1,28,28)).to(device)
        rates = get_arate_val(data)
        
        for f, l in  zip(rates, lab):
            firing_levels.append(f)
            lab_pred.append(f)        
            if l == mnist_class_anomaly:
                lab_true.append(1)
            else:
                lab_true.append(0)
                    
    fpr, tpr, threshold = metrics.roc_curve(lab_true, lab_pred)
    roc_auc = metrics.auc(fpr, tpr)
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = threshold[optimal_idx]
    fig = plt.figure(figsize =(4, 4))
    plt.title('Receiver Operating Characteristic')
    plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
    plt.legend(loc = 'lower right')
    plt.plot([0, 1], [0, 1],'r--')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.show()
    writer.add_figure('ROC', fig)

In [ ]:


def show_plot():
    #centroids = model.fuzzy[0].get_centroids().detach().cpu().numpy()
    embedings = []
    labels_expected = []
    with torch.no_grad():
        for data, target in tqdm(test_loader, desc='Encoding'):
            data = data.view((-1,1,28,28)).to(device)
            mu,_,_ = fvae.half_pass(data)
            embedings.append(mu.cpu().numpy())
            #labels_expected.append((target == mnist_class_anomaly).cpu().numpy())
            labels_expected.append(target.cpu().numpy())
    embedings = np.concatenate(embedings, axis=0)
    labels_expected = np.concatenate(labels_expected, axis=0)

    plt.figure(figsize=(18, 6))

    R, C = 1, 3

    plt.subplot(R, C, 1)
    plt.title("MNIST XY")
    
    plt.scatter(embedings[:, 0],      embedings[:,  1], c=labels_expected, cmap='tab10', s=2)
    #plt.scatter(centroids[:, 0],      centroids[:, 1], marker='1', c='black', s= 50)

    plt.subplot(R, C, 2)
    plt.title("MNIST XZ")
    plt.scatter(embedings[:, 0],      embedings[:,  2], c=labels_expected, cmap='tab10', s=2)
    #plt.scatter(centroids[:, 0],      centroids[:, 2], marker='1', c='black', s= 50)

    
show_plot()

In [ ]:
def show_item_reconstructio(ind):
    for data, trg in iter(test_loader):
        data = data.to(device)
        mu, fz = fvae.half_pass(data)
        rec_x = fvae.decoder_pass(fz)
        plt.figure(figsize=(24, 6))

        R, C = 1, 6

        plt.subplot(R, C, 1)
        plt.imshow(data[ind].cpu().squeeze())
        plt.subplot(R, C, 2)
        plt.imshow(rec_x[ind].detach().cpu().squeeze())
        
        plt.subplot(R, C, 3)
        plt.imshow((rec_x[ind] - data[ind]).abs().detach().cpu().squeeze())
        break

In [ ]:
show_item_reconstructio(4)
show_item_reconstructio(3)

In [ ]:
threshold = optimal_threshold
n = 0
fig, ax = plt.subplots(10, 10, figsize=(10, 10))
with torch.no_grad():
    for data, labels in tqdm(test_loader, desc='EMNIST VIS'):
        if n >= 100:
            break
        data = data.view((-1, 1, 28, 28)).to(device) 
        
        arate = get_arate_val(data)
        
        for i in range(data.shape[0]):
            if(arate[i] > threshold):
                img = data[i]
                ax[int(n / 10), int(n % 10)].imshow(img.view(28, 28).cpu().detach().numpy(), cmap='gray')
                ax[int(n / 10), int(n % 10)].axis('off')
                n = n + 1
                    
                if n >= 100:
                    break